# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/train.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1. Configuração

In [ ]:
# @title instala dependências

%pip install -q tensorflow keras-hub

In [ ]:
# @title Imports

import os
import glob
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import model as m
import preprocess as pi
import keras_hub

In [ ]:
# @title Configurações

IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5
LR = 1e-4
FEATURE_DIM = 768
DATASET_ROOT = "/dataset/FF"
FEATURES_CACHE = "/dataset/siglip2_features_keras.npz"

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Desativa GPU

import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

print(f"GPUs visíveis: {tf.config.list_physical_devices('GPU')}")
print(f'TensorFlow: {tf.__version__}')

## 2. Dataset FaceForensics++

No arquivo **face_extraction.ipynb** foi realizada a extração dos rostos frame por frame nos vídeos do dataset. Para melhorar a otimização, foram utilizados os seguintes filtros na extração:

- SKIP_FRAMES = 9     
_0 = todos; 9 = pula 9 (processa 1 a cada 10)_

- MAX_FRAMES_PER_VIDEO = 30          
_None = todos; ex: 100 = máx 100 frames que contenham rostos são salvos_

- MAX_READ_LIMIT = 1000                 
_None = todos; ex: 1000 = máx 1000 frames lidos por vídeo_

- MAX_VIDEOS = None                 
_None = todos; ex: 10 = máx 10 vídeos processados_

Split padrão: **70% treino / 15% validação / 15% teste**.

| Índice | Classe 
|---|---
| 0 | real 
| 1 | fake

Há também o pré-processamento para o siglip2 que é o que será usado no modelo, as imagens são salvas em 224x224 (float 32)

## 3. Funções auxiliares

In [ ]:
# @title Carrega Siglip2 sem a cabeça classificadora
def extract_features(file_paths, backbone_model, batch_size=32):
    features = []
    
    for i in range(0, len(file_paths), batch_size):
        batch_files = file_paths[i:i+batch_size]
        batch_imgs = np.array([np.load(f) for f in batch_files])
        
        batch_feats = backbone_model.get_vision_embeddings(batch_imgs)
        features.append(batch_feats.numpy())
        
    return np.concatenate(features, axis=0)

if os.path.exists("siglip2_base_patch16_224_original_backbone.keras"):
   print("Carregando backbone do SigLIP2 a partir do arquivo salvo...")
   try:
        backbone = keras.models.load_model("siglip2_base_patch16_224_original_backbone.keras")
   except Exception as e:
        print(f"Erro ao carregar o backbone localmente: {e}")
        print("Recarregando o backbone do SigLIP2 a partir do preset...")
        backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
        backbone.trainable = False
else:
  print("Baixando o backbone SigLIP2 do KerasHub...")
  backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
  backbone.trainable = False
  backbone.save("siglip2_base_patch16_224_original_backbone.keras")
  print("Salvo localmente")



In [ ]:
# @title Carrega imagens pré-processadas para o Siglip2

def load_dataset_metadata(dataset_dir):
    file_paths = []
    labels = []
    video_ids = []

    search_path = os.path.join(dataset_dir, "**", "*_siglip2.npy")
    all_files = glob.glob(search_path, recursive=True)

    for filepath in all_files:
        path_parts = filepath.split(os.sep)
        label_str = path_parts[-3].lower()
        video_id = path_parts[-2]

        label = 1 if "fake" in label_str or "manipulated" in label_str else 0

        file_paths.append(filepath)
        labels.append(label)
        video_ids.append(video_id)

    return np.array(file_paths), np.array(labels), np.array(video_ids)

file_paths, labels, video_ids = load_dataset_metadata(DATASET_ROOT)
print(f"Total de amostras encontradas: {len(file_paths)}")
print(f"Distribuição de classes - Reais (0): {np.sum(labels == 0)} | Fakes (1): {np.sum(labels == 1)}")

gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(file_paths, labels, groups=video_ids))

train_paths, train_labels = file_paths[train_idx], labels[train_idx]
val_paths, val_labels = file_paths[val_idx], labels[val_idx]


In [ ]:
print("[INFO] Extraindo embeddings para o conjunto de Treino...")
X_train = extract_features(train_paths, backbone)
print("[INFO] Extraindo embeddings para o conjunto de Validação...")
X_val = extract_features(val_paths, backbone)

y_train = train_labels
y_val = val_labels

print(f"Shape das features de Treino: {X_train.shape}")
print(f"Shape das features de Validação: {X_val.shape}")

## 4. Treino

In [ ]:
# @title Cria datasets de treino, validação e teste

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(len(X_train)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(AUTOTUNE)

In [ ]:
FEATURE_DIM = X_train.shape[1]  # Ex: 768
classifier = m.build_classifier(feature_dim=FEATURE_DIM, l2_reg=1e-4)

classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

classifier.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_auc', 
        patience=2, 
        mode='max', 
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=2, 
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'model.{epoch:02d}-{val_auc:.4f}.keras',
        monitor='val_auc',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

history = classifier.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    callbacks=callbacks
)

print(f'\nAcurácia de validação (Fase 1): {history.history["val_accuracy"][-1]:.1%}')

In [ ]:
# ==========================================
# 5. AVALIAÇÃO DO MODELO
# ==========================================
val_preds = classifier.predict(X_val).flatten()
val_preds_binary = (val_preds > 0.5).astype(int)

print("\n--- Relatório de Classificação (Validação) ---")
print(classification_report(y_val, val_preds_binary, target_names=['Real', 'Fake']))
print(f"AUC-ROC Final: {roc_auc_score(y_val, val_preds):.4f}")

In [ ]:
# # @title Constrói o modelo completo
# model = keras.Sequential([
#     keras.layers.Input(shape=(FEATURE_DIM,)),
#     keras.layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
#     keras.layers.Dropout(0.4),
#     keras.layers.Dense(1, activation='sigmoid')
# ])

# model.compile(optimizer=keras.optimizers.Adam(LR), loss='binary_crossentropy',
#               metrics=['accuracy', keras.metrics.AUC(name='auc')])
# model.summary()

# for ilayer, layer in enumerate(model.layers):
#     print("{:3.0f} {:10}".format(ilayer, layer.name))

In [ ]:
# # @title Plota métricas de treino e validação

# acc_all     = history.history['accuracy']    
# val_acc_all = history.history['val_accuracy']

# auc_all     = history.history['auc']    
# val_auc_all = history.history['val_auc']

# loss_all     = history.history['loss']
# val_loss_all = history.history['val_loss']


# # Acurácia

# plt.figure(figsize=(10, 5))
# plt.plot(acc_all,     label='Treino',     linewidth=2)
# plt.plot(val_acc_all, label='Validação',  linewidth=2)
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue')
# plt.title('Acurácia — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('Acurácia')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()


# # AUC

# plt.figure(figsize=(10, 5))
# plt.plot(auc_all,     label='Treino',     linewidth=2)
# plt.plot(val_auc_all, label='Validação',  linewidth=2)
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue')
# plt.title('AUC — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('AUC')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()


# # Loss

# plt.figure(figsize=(10, 5))
# plt.plot(loss_all,     label='Treino',     linewidth=2)
# plt.plot(val_loss_all, label='Validação',  linewidth=2)
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue')
# plt.title('Perda — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('Perda')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()




In [ ]:
# #@title Descongela as últimas 4 camadas do vision encoder
# model.trainable = True

# freeze_until = len(backbone.vision_encoder.layers) - 4
# for layer in backbone.vision_encoder.layers[:freeze_until]:
#     layer.trainable = False

# trainable = sum(1 for l in model.layers if l.trainable)
# frozen    = sum(1 for l in model.layers if not l.trainable)
# print(f'Camadas treináveis na base: {trainable}')
# print(f'Camadas congeladas na base: {frozen}')

# trainable_ve = sum(1 for l in backbone.vision_encoder.layers if l.trainable)
# frozen_ve    = sum(1 for l in backbone.vision_encoder.layers if not l.trainable)
# print(f'Vision encoder — Treináveis: {trainable_ve}, Congeladas: {frozen_ve}')

# backbone.save("siglip2_base_patch16_224_finetune_backbone.keras")


In [ ]:
# @title Re-compila com taxa de aprendizado menor — Fase 2

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary(show_trainable=True)

In [ ]:
# #@title Treinamento Fase 2 (5 epochs, LR=1e-5)

# EPOCHS_2 = 5

# history2 = model.fit(
#     train_ds,
#     epochs=EPOCHS + EPOCHS_2,
#     initial_epoch=EPOCHS,
#     validation_data=val_ds,
#     verbose=1,
#     callbacks=callbacks
# )

# print(f'\nAcurácia de validação (Fase 2): {history2.history["val_accuracy"][-1]:.1%}')

In [ ]:
# # @title Concatena históricos e plota
# acc_all     = history.history['accuracy']     + history2.history['accuracy']
# val_acc_all = history.history['val_accuracy'] + history2.history['val_accuracy']

# plt.figure(figsize=(10, 5))
# plt.plot(acc_all,     label='Treino',     linewidth=2)
# plt.plot(val_acc_all, label='Validação',  linewidth=2)
# plt.axvline(x=EPOCHS - 0.5, color='gray', linestyle='--', linewidth=1.5,
#             label='Início do fine-tuning')
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue',  label='Extração de características')
# plt.fill_betweenx([0, 1], EPOCHS - 0.5, len(acc_all),
#                   alpha=0.05, color='orange', label='Fine-tuning')
# plt.title('Acurácia — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('Acurácia')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()



# auc_all     = history.history['auc']     + history2.history['auc']
# val_auc_all = history.history['val_auc'] + history2.history['val_auc']

# plt.figure(figsize=(10, 5))
# plt.plot(auc_all,     label='Treino',     linewidth=2)
# plt.plot(val_auc_all, label='Validação',  linewidth=2)
# plt.axvline(x=EPOCHS - 0.5, color='gray', linestyle='--', linewidth=1.5,
#             label='Início do fine-tuning')
# plt.fill_betweenx([0, 1], 0, EPOCHS - 0.5,
#                   alpha=0.05, color='blue',  label='Extração de características')
# plt.fill_betweenx([0, 1], EPOCHS - 0.5, len(auc_all),
#                   alpha=0.05, color='orange', label='Fine-tuning')
# plt.title('AUC — Transfer Learning no FF++', fontsize=13)
# plt.xlabel('Época')
# plt.ylabel('AUC')
# plt.legend(loc='lower right')
# plt.ylim(0, 1)
# plt.tight_layout()
# plt.show()